# pyqtcm1 in the cloud

Run the Neelin–Zeng Quasi-Equilibrium Tropical Circulation Model (QTCM1) in your browser.
The repository is self-contained — code, boundary data and Q-flux all install in one step.

Docs: https://pyqtcm1.readthedocs.io

In [ ]:
import os
if os.path.isdir('pyqtcm1'):                 # re-run: update the clone
    !git -C pyqtcm1 pull --ff-only
else:
    !git clone --depth 1 https://github.com/AndrewILWilliams/pyqtcm1.git
%pip install -q -e ./pyqtcm1[numba]
import sys; sys.path.insert(0, 'pyqtcm1/src')

## A one-year control run

Cold start, climatological seasonal SST, daily-mean precipitation and
monthly means archived as xarray Datasets (~1–2 min/simulated year on a
Colab CPU; the first day includes Numba compilation).

In [ ]:
from qtcm1.config import RunConfig
from qtcm1.driver import ControlRun

cfg = RunConfig(output={
    'Ts': {'freq': 'monthly', 'kind': 'mean'},
    'T1': {'freq': 'monthly', 'kind': 'mean'},
    'Qc': {'freq': 'daily',   'kind': 'mean'},
})
run = ControlRun(config=cfg)
run.run_years(1, progress=lambda d, date: print(f'day {d}', end='\r'))
dsets = run.to_datasets()
dsets['monthly']

In [ ]:
import matplotlib.pyplot as plt

prec = dsets['daily']['Qc'] / 28.125          # W/m2 -> mm/day
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
prec.isel(time=slice(-90, None)).mean('time').plot(
    ax=ax[0], cmap='viridis', cbar_kwargs={'label': 'mm/day'})
ax[0].set_title('Precipitation (last 90 days)')
prec.sel(lat=slice(-15, 15)).mean(('lat', 'lon')).plot(ax=ax[1])
ax[1].set_title('Tropical-mean precipitation')
plt.tight_layout()

## Try an experiment

Everything in the [example gallery](https://pyqtcm1.readthedocs.io/en/latest/examples.html)
runs here unchanged — SST anomalies, greenhouse forcing, the slab ocean,
bit-exact restarts. For instance, a +2 K equatorial-Pacific warm patch:

In [ ]:
# %load pyqtcm1/examples/02_sst_anomaly.py   # uncomment to load & run